In [21]:
import pandas as pd
from pathlib import Path

BASE = Path.cwd().parent
RAW  = BASE / "data" / "raw"
PROCESSED = (Path.cwd().parent/"data"/"processed")
PROCESSED.mkdir(parents=True, exist_ok=True)



In [22]:
nav = pd.read_csv(RAW / "02_nav_history.csv")
nav["date"] = pd.to_datetime(nav["date"])
nav = nav.drop_duplicates(subset=["amfi_code", "date"])
#invalid nav
nav = nav[nav["nav"] > 0]  

full_bdays= pd.bdate_range(nav["date"].min(),nav["date"].max())

clean_parts=[]


for code,grp in nav.groupby("amfi_code"):
    grp=grp.set_index("date").reindex(full_bdays)
    grp["nav"] = grp["nav"].ffill()     
    grp["amfi_code"] = code
    grp.index.name= "date"
    clean_parts.append(grp.reset_index())




In [23]:
clean_nav = pd.concat(clean_parts, ignore_index=True)
# Compute daily return AFTER cleaning
clean_nav["daily_return_pct"] = (
    clean_nav.sort_values("date")
             .groupby("amfi_code")["nav"]
             .pct_change() * 100
)
clean_nav.to_csv(PROCESSED / "clean_nav.csv", index=False)
print(f"clean_nav: {clean_nav.shape}")



clean_nav: (46000, 4)


In [24]:
#investor transactions cleaning

tx = pd.read_csv(RAW / "08_investor_transactions.csv")
tx["transaction_date"] = pd.to_datetime(tx["transaction_date"])
tx["transaction_type"] = tx["transaction_type"].str.strip().str.title()
assert tx["transaction_type"].isin(["Sip","Lumpsum","Redemption"]).all()
tx = tx[tx["amount_inr"] > 0]
tx = tx.drop_duplicates()
assert tx["kyc_status"].isin(["Verified","Pending"]).all()
tx.to_csv(PROCESSED / "clean_transactions.csv", index=False)
print(f"Transactions cleaned: {tx.shape}")

Transactions cleaned: (32778, 13)


In [25]:
#scheme performance cleaning

perf = pd.read_csv(RAW / "07_scheme_performance.csv")

# expense ratios- SEBI allows 0.1% - 2.5%
bad_er = perf[~perf["expense_ratio_pct"].between(0.1, 2.5)]
if not bad_er.empty:
    print(f"Warning: {len(bad_er)} rows outside expense_ratio range")
# Flag negative Sharpe (possible but worth logging)
neg_sharpe = perf[perf["sharpe_ratio"] < 0]
print(f"Funds with negative Sharpe: {len(neg_sharpe)}")
perf.to_csv(PROCESSED / "clean_performance.csv", index=False)

Funds with negative Sharpe: 0


In [26]:
#Fund master

master = pd.read_csv(RAW / "01_fund_master.csv")
master["amfi_code"] = master["amfi_code"].astype(str)
master["launch_date"] = pd.to_datetime(master["launch_date"], errors="coerce")
str_cols = master.select_dtypes("object").columns
master[str_cols] = master[str_cols].apply(lambda s: s.str.strip())
master.to_csv(PROCESSED / "clean_fund_master.csv", index=False)
print(f"clean_fund_master: {master.shape}")


clean_fund_master: (40, 15)


C:\Users\MANAVI\AppData\Local\Temp\ipykernel_15500\188075030.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = master.select_dtypes("object").columns


In [ ]:
aum = pd.read_csv(RAW / "03_aum_by_fund_house.csv")
aum["date"] = pd.to_datetime(aum["date"], errors="coerce")
aum.to_csv(PROCESSED / "clean_aum.csv", index=False)
print(f"clean_aum: {aum.shape}")

         date           fund_house  aum_lakh_crore  aum_crore  num_schemes
0  2022-03-31      SBI Mutual Fund            6.05     605000          186
1  2022-03-31  ICICI Prudential MF            4.65     465000          216
2  2022-03-31     HDFC Mutual Fund            4.35     435000          195
3  2022-03-31      Nippon India MF            2.70     270000          177
4  2022-03-31    Kotak Mahindra MF            2.70     270000          168
clean_aum: (90, 5)


In [30]:
#monthly sip inflows

sip = pd.read_csv(RAW / "04_monthly_sip_inflows.csv")
sip["month"] = pd.to_datetime(sip["month"], format="%Y-%m", errors="coerce")
sip.to_csv(PROCESSED / "clean_sip_inflows.csv", index=False)
print(f"clean_sip_inflows: {sip.shape}")

clean_sip_inflows: (48, 6)


In [31]:
ci = pd.read_csv(RAW / "05_category_inflows.csv")
ci["month"] = pd.to_datetime(ci["month"], format="%Y-%m", errors="coerce")
ci.to_csv(PROCESSED / "clean_category_inflows.csv", index=False)
print(f"clean_category_inflows: {ci.shape}")

clean_category_inflows: (144, 3)


In [ ]:
#folio counts

folio = pd.read_csv(RAW / "06_industry_folio_count.csv")
folio["date"] = pd.to_datetime(folio["date"], errors="coerce")
folio = folio.sort_values("date").reset_index(drop=True)
folio.to_csv(PROCESSED / "clean_folio_count.csv", index=False)
print(f"clean_folio_count: {folio.shape}")


In [33]:
#Portfolio holdings 
hold = pd.read_csv(RAW / "09_portfolio_holdings.csv")
hold["amfi_code"] = hold["amfi_code"].astype(str)
hold.to_csv(PROCESSED / "clean_portfolio.csv", index=False)
print(f"clean_portfolio: {hold.shape}")

clean_portfolio: (322, 8)
